# Streaming ingestion with Amazon Kinesis

This portfolio walkthrough accompanies the completed lab. It demonstrates the producer → stream → consumer pattern using the scripts in `src/cli/`.

It assumes that the lab Kinesis stream already exists and that AWS credentials are configured through the normal boto3 credential chain. It does not create billable AWS resources.

## 1. Inspect the sample event

The event describes a customer session and a small browsing history. The example JSON is in `data/example_record.json`.

In [ ]:
import json
from pathlib import Path

record = json.loads(Path("data/example_record.json").read_text())
record

## 2. Produce a record

The producer sends JSON with Kinesis `PutRecord`. The session ID is used as the partition key.

In [ ]:
import boto3

stream_name = "YOUR_SOURCE_STREAM_NAME"  # replace with your lab stream
client = boto3.client("kinesis")
response = client.put_record(
    StreamName=stream_name,
    Data=json.dumps(record).encode("utf-8"),
    PartitionKey=record["session_id"],
)
print(response["ShardId"], response["SequenceNumber"])

## 3. Consume records

Run `src/cli/consumer_from_cli.py` in a separate terminal while sending events from the producer. The consumer discovers shards, obtains iterators, and polls `GetRecords`.

## Key ideas

- Producer and consumer run independently.
- Records are distributed to shards using a partition key.
- The consumer keeps the next shard iterator to continue reading.